In [1]:
#!pip install --upgrade kaleido

In [2]:
from datetime import datetime
from pathlib import Path
import platform, sys


DOMAIN = "ts"             # "ts" | "corr" | "geo"
DATASET_NAME = "demo"     # napr. "noaa_demo" | "wdi_demo" | "osm_sk_demo"
LIBRARY = "plotly"        # "matplotlib" | "seaborn" | "plotly" | "geopandas_matplotlib"
TASK_ID = "trend_a_sezonnost"
N_RUNS = 7
SEED = 42
TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [3]:
import os, time, json, csv
import numpy as np, pandas as pd, statistics as stats
import matplotlib, matplotlib.pyplot as plt, seaborn as sns
import plotly
import plotly.express as px, plotly.graph_objects as go


try:
    import geopandas as gpd
except Exception:
    gpd = None
try:
    import yaml
except Exception:
    yaml = None

plt.rcParams.update({
    "figure.dpi": 120, "figure.figsize": (8, 4.5),
    "axes.titlesize": 12, "axes.labelsize": 10,
    "legend.fontsize": 9, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "font.family": "DejaVu Sans"
})
sns.set_theme(style="whitegrid")
px.defaults.template = "plotly_white"
px.defaults.width = 800
px.defaults.height = 450

In [4]:
def _try_load_yaml(path: Path):
    if yaml is None or not path.exists():
        return None
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

PROJECT_ROOT = Path.cwd()
CFG_PATHS  = _try_load_yaml(PROJECT_ROOT / "configs" / "paths.yaml") or {}
CFG_PLOTS  = _try_load_yaml(PROJECT_ROOT / "configs" / "plots.yaml") or {}
CFG_WEIGHTS= _try_load_yaml(PROJECT_ROOT / "configs" / "weights.yaml") or {}

OUT_ROOT   = Path(CFG_PATHS.get("outputs", "practicka_cast/outputs"))
FIG_DIR    = OUT_ROOT / "figures"
HTML_DIR   = OUT_ROOT / "html"
METRICS_DIR= OUT_ROOT / "metrics"
TABLES_DIR = OUT_ROOT / "tables"
for d in [FIG_DIR, HTML_DIR, METRICS_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DOMAIN_MAP = {"ts":"time_series","corr":"correlations","geo":"geo"}
DOMAIN_DIR = DOMAIN_MAP.get(DOMAIN, DOMAIN)
LIB_DIR    = LIBRARY.replace("_","")
FIG_SUBDIR = FIG_DIR / DOMAIN_DIR / LIB_DIR
HTML_SUBDIR= HTML_DIR / DOMAIN_DIR
FIG_SUBDIR.mkdir(parents=True, exist_ok=True)
HTML_SUBDIR.mkdir(parents=True, exist_ok=True)

def file_basename(prefix: str, variant: str, ext: str) -> Path:
    name = f"{DOMAIN}_{LIBRARY}_{DATASET_NAME}_{TASK_ID}_{variant}_{TIMESTAMP}.{ext}"
    return (HTML_SUBDIR if ext.lower()=="html" else FIG_SUBDIR) / name

class Timer:
    def __enter__(self): self.t0 = time.perf_counter(); return self
    def __exit__(self, *args): self.elapsed = (time.perf_counter()-self.t0)*1000.0

def file_size_kb(path: Path):
    try: return os.path.getsize(path)/1024.0
    except OSError: return None

def write_metrics(row: dict, csv_path: Path = METRICS_DIR / "metrics.csv"):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    exists = csv_path.exists()
    with open(csv_path, "a", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists: w.writeheader()
        w.writerow(row)

def count_loc_from_callable(fn) -> int:
    import inspect, textwrap
    try:
        src = textwrap.dedent(inspect.getsource(fn))
        lines = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
        return len(lines)
    except Exception:
        return None


def runtime_meta():
   return {
       "python": sys.version.split()[0],
       "platform": platform.platform(),
       "matplotlib": matplotlib.__version__,
       "seaborn": sns.__version__,
       "plotly": plotly.__version__
   }

In [5]:
np.random.seed(SEED)

def load_data(domain: str, dataset_name: str):
    if domain == "ts":
        idx = pd.date_range("2020-01-01", periods=60, freq="MS")
        base = np.sin(np.linspace(0, 8*np.pi, len(idx)))*3 + np.linspace(0,10,len(idx))
        noise = np.random.normal(0,0.7,size=len(idx))
        return pd.DataFrame({"date": idx, "value": base + noise})
    elif domain == "corr":
        rng = np.random.default_rng(SEED)
        x = rng.normal(0,1,1000); y = 0.7*x + rng.normal(0,0.7,1000); z = rng.normal(0,1,1000)
        return pd.DataFrame({"x":x,"y":y,"z":z})
    elif domain == "geo":
        if gpd is None: raise RuntimeError("Geopandas nie je dostupný.")
        pts = pd.DataFrame({"lon": 17.10+np.random.rand(300)*0.3, "lat": 48.10+np.random.rand(300)*0.3,
                            "cat": np.random.choice(["A","B","C"], size=300)})
        return gpd.GeoDataFrame(pts, geometry=gpd.points_from_xy(pts["lon"], pts["lat"]), crs="EPSG:4326")
    else:
        raise ValueError(f"Neznáma doména: {domain}")

DF = load_data(DOMAIN, DATASET_NAME)
DF.head()

,date,value
0,2020-01-01,0.347700
1,2020-02-01,1.312343
2,2020-03-01,3.050077
3,2020-04-01,4.446862
4,2020-05-01,3.487517


In [6]:
def plot_baseline(df):
    if LIBRARY == "matplotlib":
        fig, ax = plt.subplots()
        if DOMAIN == "ts":
            ax.plot(df["date"], df["value"], lw=1.5, color="#1f77b4")
            ax.set_title("Časový rad – baseline"); ax.set_xlabel("Dátum"); ax.set_ylabel("Hodnota")
            fig.autofmt_xdate()
        elif DOMAIN == "corr":
            ax.scatter(df["x"], df["y"], s=8, alpha=0.6); ax.set_title("Scatter x–y – baseline")
        elif DOMAIN == "geo":
            base = df.plot(figsize=(6,6), markersize=5, alpha=0.6); fig = base.get_figure()
        return fig, "mpl"

    if LIBRARY == "seaborn":
        fig, ax = plt.subplots()
        if DOMAIN == "ts":
            sns.lineplot(df, x="date", y="value", ax=ax); ax.set_title("Časový rad – baseline (Seaborn)"); fig.autofmt_xdate()
        elif DOMAIN == "corr":
            sns.scatterplot(df, x="x", y="y", s=15, alpha=0.6, ax=ax); ax.set_title("Scatter x–y – baseline (Seaborn)")
        return fig, "mpl"

    if LIBRARY == "plotly":
        if DOMAIN == "ts":
            fig = px.line(df, x="date", y="value", title="Časový rad – baseline (Plotly)")
        elif DOMAIN == "corr":
            fig = px.scatter(df, x="x", y="y", opacity=0.7, title="Scatter x–y – baseline (Plotly)")
        elif DOMAIN == "geo":
            fig = px.scatter_geo(df, lon="lon", lat="lat", color="cat", opacity=0.7, title="Bodová mapa – baseline (Plotly)")
        return fig, "plotly"

    raise ValueError(f"Nepodporovaná knižnica: {LIBRARY}")

In [7]:
def plot_final(df):
    if LIBRARY in ("matplotlib","seaborn"):
        fig, ax = plt.subplots()
        if DOMAIN == "ts":
            w = 6; df2 = df.copy(); df2["trend"] = df2["value"].rolling(w, min_periods=1, center=True).mean()
            ax.plot(df2["date"], df2["value"], color="#1f77b4", lw=1.0, alpha=0.5, label="Hodnota")
            ax.plot(df2["date"], df2["trend"], color="#d62728", lw=2.0, label=f"Trend (MA{w})")
            ax.set_title("Vývoj indikátora s trendom"); ax.set_xlabel("Dátum"); ax.set_ylabel("Hodnota [jednotky]")
            ax.legend(); ax.grid(True, alpha=0.3); fig.autofmt_xdate()
        elif DOMAIN == "corr":
            sns.kdeplot(df, x="x", y="y", levels=6, fill=True, cmap="Blues", alpha=0.7, ax=ax)
            sns.scatterplot(df.sample(min(500, len(df)), random_state=SEED), x="x", y="y", s=8, alpha=0.4, color="#2ca02c", ax=ax)
            ax.set_title("Závislosť x–y s hustotnými obálkami"); ax.grid(True, alpha=0.3)
        elif DOMAIN == "geo":
            if gpd is None: raise RuntimeError("Geopandas nie je dostupný.")
            base = df.plot(figsize=(7,7), alpha=0.7, markersize=6, column="cat", legend=True, categorical=True)
            ax = base.axes; ax.set_title("Bodová mapa kategórií"); ax.set_axis_off(); fig = base.get_figure()
        return fig, "mpl"

    if LIBRARY == "plotly":
        if DOMAIN == "ts":
            df2 = df.copy(); df2["trend"] = df2["value"].rolling(6, min_periods=1, center=True).mean()
            fig = go.Figure()
            fig.add_trace(go.Scatter(x=df2["date"], y=df2["value"], mode="lines", name="Hodnota", line=dict(color="#1f77b4", width=1)))
            fig.add_trace(go.Scatter(x=df2["date"], y=df2["trend"], mode="lines", name="Trend (MA6)", line=dict(color="#d62728", width=2)))
            fig.update_layout(title="Vývoj indikátora s trendom", xaxis_title="Dátum", yaxis_title="Hodnota [jednotky]",
                              hovermode="x unified", legend=dict(orientation="h", y=-0.2))
        elif DOMAIN == "corr":
            fig = px.scatter(df, x="x", y="y", opacity=0.5, trendline="ols", trendline_color_override="#d62728",
                             labels={"x":"x","y":"y"}, title="Závislosť x–y s trendovou čiarou")
            fig.update_traces(marker=dict(size=6))
        elif DOMAIN == "geo":
            fig = px.scatter_geo(df, lon="lon", lat="lat", color="cat", opacity=0.75, title="Bodová mapa kategórií",
                                 labels={"lon":"Zemepisná dĺžka","lat":"Zemepisná šírka"})
            fig.update_geos(fitbounds="locations", visible=False); fig.update_layout(legend=dict(orientation="h", y=-0.2))
        return fig, "plotly"

    raise ValueError(f"Nepodporovaná knižnica: {LIBRARY}")

In [8]:
def render_and_time(make_fig_callable, df, variant: str, n_runs: int = 5) -> dict:
    times, out_files = [], []

    for _ in range(n_runs):
        with Timer() as t:
            fig, kind = make_fig_callable(df)
            if kind == "mpl":
                fig.canvas.draw(); plt.close(fig)
            elif kind == "plotly":
                _ = fig.to_dict()
        times.append(t.elapsed)

    fig, kind = make_fig_callable(df)
    export_elapsed_ms, fsize_kb, saved_path = None, None, None

    if kind == "mpl":
        saved_path = file_basename("plot", variant, "png")
        with Timer() as t:
            fig.savefig(saved_path, dpi=CFG_PLOTS.get("dpi", 120), bbox_inches="tight")
        export_elapsed_ms = t.elapsed; fsize_kb = file_size_kb(saved_path); plt.close(fig)
        if CFG_PLOTS.get("svg", True):
            svg_path = saved_path.with_suffix(".svg")
            fig, _ = make_fig_callable(df); fig.savefig(svg_path, bbox_inches="tight"); plt.close(fig)
            out_files.append(svg_path)
        out_files.append(saved_path)

    elif kind == "plotly":
        saved_path = file_basename("plot", variant, "html")
        with Timer() as t:
            fig.write_html(saved_path, include_plotlyjs="cdn", full_html=True)
        export_elapsed_ms = t.elapsed; fsize_kb = file_size_kb(saved_path); out_files.append(saved_path)
        try:
            png_path = file_basename("plot", variant, "png")
            fig.write_image(png_path, scale=2); out_files.append(png_path)
            if CFG_PLOTS.get("svg", True):
                svg_path = file_basename("plot", variant, "svg")
                fig.write_image(svg_path); out_files.append(svg_path)
        except Exception as e:
            print("Upozornenie: PNG/SVG export (Plotly) preskočený:", e)

    return {
        "render_ms_median": stats.median(times),
        "render_ms_p95": np.percentile(times, 95),
        "export_ms": export_elapsed_ms,
        "file_size_kb": fsize_kb,
        "saved": str(saved_path),
        "all_files": [str(p) for p in out_files]
    }

loc_baseline = count_loc_from_callable(plot_baseline)
loc_final    = count_loc_from_callable(plot_final)
res_baseline = render_and_time(plot_baseline, DF, "baseline", n_runs=N_RUNS)
res_final    = render_and_time(plot_final,    DF, "final",    n_runs=N_RUNS)

meta = runtime_meta()
common = {
    "timestamp": TIMESTAMP, "domain": DOMAIN, "dataset": DATASET_NAME, "library": LIBRARY, "task_id": TASK_ID,
    "python": meta["python"], "platform": meta["platform"],
    "matplotlib_ver": meta["matplotlib"], "seaborn_ver": meta["seaborn"], "plotly_ver": meta["plotly"]
}

write_metrics({**common, "variant":"baseline",
               "render_ms_median": round(res_baseline["render_ms_median"],2),
               "render_ms_p95":   round(res_baseline["render_ms_p95"],2),
               "export_ms":       round(res_baseline["export_ms"] or 0.0,2),
               "loc": loc_baseline,
               "file_size_kb":    round(res_baseline["file_size_kb"] or 0.0,2),
               "saved": res_baseline["saved"]})

write_metrics({**common, "variant":"final",
               "render_ms_median": round(res_final["render_ms_median"],2),
               "render_ms_p95":   round(res_final["render_ms_p95"],2),
               "export_ms":       round(res_final["export_ms"] or 0.0,2),
               "loc": loc_final,
               "file_size_kb":    round(res_final["file_size_kb"] or 0.0,2),
               "saved": res_final["saved"]})

print("Hotovo. Uložené:")
print("  baseline:", res_baseline["saved"])
print("  final   :", res_final["saved"])
print("Metriky ->", METRICS_DIR / "metrics.csv")

Hotovo. Uložené:
  baseline: practicka_cast\outputs\html\time_series\ts_plotly_demo_trend_a_sezonnost_baseline_2026-03-23_19-19-29.html
  final   : practicka_cast\outputs\html\time_series\ts_plotly_demo_trend_a_sezonnost_final_2026-03-23_19-19-29.html
Metriky -> practicka_cast\outputs\metrics\metrics.csv


In [9]:
m = pd.read_csv(METRICS_DIR / "metrics.csv")
subset = m[(m["timestamp"] == TIMESTAMP) & (m["dataset"] == DATASET_NAME) & (m["library"] == LIBRARY)]
subset[["variant","render_ms_median","render_ms_p95","export_ms","loc","file_size_kb","saved"]]

,variant,render_ms_median,render_ms_p95,export_ms,loc,file_size_kb,saved
0,baseline,53.77,390.97,35.86,28,10.01,practicka_cast\outputs\html\time_series\ts_plo...
1,final,11.83,13.95,53.15,36,11.84,practicka_cast\outputs\html\time_series\ts_plo...
